# T37 - 同花顺Excel导入财务指标

## 6. 可视化展示

本章展示财务指标的可视化分析方法。

## 6.1 环境准备

In [ ]:
# 导入标准库
import os
import sys
from datetime import datetime
from pathlib import Path

# 添加项目路径
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

# 导入第三方库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 设置样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("可视化环境准备完成")

## 6.2 示例数据准备

In [ ]:
# 创建示例财务数据
np.random.seed(42)

# 时间序列
dates = pd.date_range(start='2020-01-01', end='2023-12-31', freq='Q')

# 示例数据
sample_data = pd.DataFrame({
    'dt': dates,
    'thscode': ['101660001.IB'] * len(dates),
    'ths_total_assets_bond': np.random.uniform(800000, 1200000, len(dates)),
    'ths_total_liab_bond': np.random.uniform(400000, 700000, len(dates)),
    'ths_total_current_assets_bond': np.random.uniform(300000, 500000, len(dates)),
    'ths_total_current_liab_bond': np.random.uniform(150000, 300000, len(dates)),
    'ths_total_owner_equity_bond': np.random.uniform(300000, 500000, len(dates)),
    'operating_revenue': np.random.uniform(400000, 600000, len(dates)),
    'net_profit': np.random.uniform(50000, 150000, len(dates)),
})

# 计算财务比率
sample_data['debt_ratio'] = sample_data['ths_total_liab_bond'] / sample_data['ths_total_assets_bond']
sample_data['current_ratio'] = sample_data['ths_total_current_assets_bond'] / sample_data['ths_total_current_liab_bond']
sample_data['net_margin'] = sample_data['net_profit'] / sample_data['operating_revenue']

print("示例数据:")
display(sample_data.head())

## 6.3 财务结构分析

### 6.3.1 资产负债结构饼图

In [ ]:
def plot_balance_sheet_structure(df, title='资产负债结构'):
    """
    绘制资产负债结构饼图
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    title : str
        图表标题
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 资产结构
    assets_data = [
        df['ths_total_current_assets_bond'].iloc[-1],
        df['ths_total_assets_bond'].iloc[-1] - df['ths_total_current_assets_bond'].iloc[-1]
    ]
    axes[0].pie(assets_data, labels=['流动资产', '非流动资产'], autopct='%1.1f%%',
                colors=['#2ecc71', '#3498db'], startangle=90)
    axes[0].set_title('资产结构')
    
    # 负债和权益结构
    liabilities_data = [
        df['ths_total_liab_bond'].iloc[-1],
        df['ths_total_owner_equity_bond'].iloc[-1]
    ]
    axes[1].pie(liabilities_data, labels=['负债', '所有者权益'], autopct='%1.1f%%',
                colors=['#e74c3c', '#9b59b6'], startangle=90)
    axes[1].set_title('负债与权益结构')
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制资产负债结构
plot_balance_sheet_structure(sample_data, '示例企业资产负债结构')

### 6.3.2 资产负债规模趋势

In [ ]:
def plot_balance_trend(df, title='资产负债规模趋势'):
    """
    绘制资产负债规模趋势图
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    title : str
        图表标题
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(df['dt'], df['ths_total_assets_bond'] / 10000, 'b-o', label='总资产', linewidth=2)
    ax.plot(df['dt'], df['ths_total_liab_bond'] / 10000, 'r-s', label='总负债', linewidth=2)
    ax.plot(df['dt'], df['ths_total_owner_equity_bond'] / 10000, 'g-^', label='所有者权益', linewidth=2)
    
    ax.set_xlabel('报告期')
    ax.set_ylabel('金额（万元）')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# 绘制趋势图
plot_balance_trend(sample_data)

## 6.4 财务比率分析

### 6.4.1 偿债能力指标

In [ ]:
def plot_solvency_ratios(df, title='偿债能力指标趋势'):
    """
    绘制偿债能力指标趋势图
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    title : str
        图表标题
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 负债比率
    axes[0].plot(df['dt'], df['debt_ratio'], 'b-o', linewidth=2)
    axes[0].axhline(y=0.6, color='orange', linestyle='--', label='预警线(60%)')
    axes[0].axhline(y=0.8, color='red', linestyle='--', label='危险线(80%)')
    axes[0].set_xlabel('报告期')
    axes[0].set_ylabel('负债比率')
    axes[0].set_title('负债比率趋势')
    axes[0].legend(loc='best')
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)
    
    # 流动比率
    axes[1].plot(df['dt'], df['current_ratio'], 'g-s', linewidth=2)
    axes[1].axhline(y=1.5, color='orange', linestyle='--', label='预警线(1.5)')
    axes[1].axhline(y=1.0, color='red', linestyle='--', label='危险线(1.0)')
    axes[1].set_xlabel('报告期')
    axes[1].set_ylabel('流动比率')
    axes[1].set_title('流动比率趋势')
    axes[1].legend(loc='best')
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制偿债能力指标
plot_solvency_ratios(sample_data)

### 6.4.2 盈利能力指标

In [ ]:
def plot_profitability(df, title='盈利能力趋势'):
    """
    绘制盈利能力趋势图
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    title : str
        图表标题
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 收入和利润趋势
    axes[0].bar(df['dt'].dt.strftime('%Y-Q%q'), df['operating_revenue'] / 10000, 
                color='#3498db', alpha=0.7, label='营业收入')
    axes[0].plot(df['dt'].dt.strftime('%Y-Q%q'), df['net_profit'] / 10000, 
                 'r-o', linewidth=2, label='净利润')
    axes[0].set_xlabel('报告期')
    axes[0].set_ylabel('金额（万元）')
    axes[0].set_title('收入与利润趋势')
    axes[0].legend(loc='best')
    axes[0].tick_params(axis='x', rotation=45)
    
    # 净利率趋势
    axes[1].plot(df['dt'], df['net_margin'] * 100, 'g-^', linewidth=2, markersize=8)
    axes[1].set_xlabel('报告期')
    axes[1].set_ylabel('净利率（%）')
    axes[1].set_title('净利率趋势')
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制盈利能力
plot_profitability(sample_data)

## 6.5 综合评分雷达图

In [ ]:
def plot_radar_chart(scores, title='财务综合评分'):
    """
    绘制财务评分雷达图
    
    Parameters:
    -----------
    scores : dict
        各维度评分
    title : str
        图表标题
    """
    # 准备数据
    categories = ['偿债能力', '盈利能力', '运营效率']
    values = [
        scores.get('solvency_score', 0),
        scores.get('profitability_score', 0),
        scores.get('efficiency_score', 0)
    ]
    
    # 计算角度
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    values += values[:1]  # 闭合
    angles += angles[:1]  # 闭合
    
    # 绘图
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    ax.fill(angles, values, color='#3498db', alpha=0.25)
    ax.plot(angles, values, 'b-o', linewidth=2)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=12)
    ax.set_ylim(0, 40)  # 每个维度最高40分
    
    plt.title(title, fontsize=14, fontweight='bold', y=1.08)
    plt.tight_layout()
    plt.show()

# 示例评分数据
sample_scores = {
    'solvency_score': 32,
    'profitability_score': 25,
    'efficiency_score': 28,
    'total_score': 85,
    'rating': '优秀'
}

# 绘制雷达图
plot_radar_chart(sample_scores, '示例企业财务综合评分')
print(f"总分: {sample_scores['total_score']}, 评级: {sample_scores['rating']}")

## 6.6 财务指标对比分析

In [ ]:
def plot_comparison_bar(df, indicators, title='财务指标对比'):
    """
    绘制多期财务指标对比柱状图
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    indicators : list
        指标列表
    title : str
        图表标题
    """
    # 选择最近4期数据
    df_recent = df.tail(4)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(indicators))
    width = 0.2
    
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    
    for i, (idx, row) in enumerate(df_recent.iterrows()):
        values = [row.get(ind, 0) for ind in indicators]
        ax.bar(x + i * width, values, width, 
               label=row['dt'].strftime('%Y-Q%q'), color=colors[i])
    
    ax.set_xlabel('指标')
    ax.set_ylabel('值')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(indicators, rotation=45, ha='right')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

# 绘制对比图
indicators = ['debt_ratio', 'current_ratio', 'net_margin']
plot_comparison_bar(sample_data, indicators, '最近4期财务指标对比')

## 6.7 财务热力图

In [ ]:
def plot_financial_heatmap(df, title='财务指标相关性热力图'):
    """
    绘制财务指标相关性热力图
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    title : str
        图表标题
    """
    # 选择数值列
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df_numeric = df[numeric_cols]
    
    # 计算相关性矩阵
    corr_matrix = df_numeric.corr()
    
    # 绘制热力图
    fig, ax = plt.subplots(figsize=(10, 8))
    
    sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0,
                fmt='.2f', ax=ax, square=True)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制热力图
plot_financial_heatmap(sample_data)

## 6.8 仪表盘综合展示

In [ ]:
def plot_dashboard(df, scores, title='财务分析仪表盘'):
    """
    绘制财务分析仪表盘
    
    Parameters:
    -----------
    df : DataFrame
        财务数据
    scores : dict
        评分数据
    title : str
        图表标题
    """
    fig = plt.figure(figsize=(16, 12))
    
    # 1. 资产负债趋势（左上）
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(df['dt'], df['ths_total_assets_bond'] / 10000, 'b-o', label='总资产')
    ax1.plot(df['dt'], df['ths_total_liab_bond'] / 10000, 'r-s', label='总负债')
    ax1.set_title('资产负债趋势')
    ax1.legend(loc='best')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # 2. 财务比率（右上）
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.plot(df['dt'], df['debt_ratio'], 'b-o', label='负债比率')
    ax2.plot(df['dt'], df['current_ratio'], 'g-s', label='流动比率')
    ax2.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5)
    ax2.set_title('财务比率趋势')
    ax2.legend(loc='best')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # 3. 盈利能力（左下）
    ax3 = fig.add_subplot(2, 2, 3)
    ax3.bar(range(len(df)), df['operating_revenue'] / 10000, color='#3498db', alpha=0.7, label='营业收入')
    ax3.plot(range(len(df)), df['net_profit'] / 10000, 'r-o', linewidth=2, label='净利润')
    ax3.set_xticks(range(len(df)))
    ax3.set_xticklabels([d.strftime('%Y-Q%q') for d in df['dt']], rotation=45)
    ax3.set_title('收入与利润')
    ax3.legend(loc='best')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. 评分卡片（右下）
    ax4 = fig.add_subplot(2, 2, 4)
    ax4.axis('off')
    
    score_text = f"""
    综合财务评分
    ━━━━━━━━━━━━━━━━━━━
    
    偿债能力: {scores.get('solvency_score', 0):.0f} / 40
    盈利能力: {scores.get('profitability_score', 0):.0f} / 30
    运营效率: {scores.get('efficiency_score', 0):.0f} / 30
    
    ━━━━━━━━━━━━━━━━━━━
    总分: {scores.get('total_score', 0):.0f} / 100
    评级: {scores.get('rating', 'N/A')}
    """
    ax4.text(0.5, 0.5, score_text, transform=ax4.transAxes,
             fontsize=14, verticalalignment='center', horizontalalignment='center',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 绘制仪表盘
plot_dashboard(sample_data, sample_scores)